### Transform Refunds Data
 1. Extract specific portion of the string from refund_reason using split function
 2. Extract specific portion of the string from refund_reason using regexp_extract function
 3. Extract date and time from refund_timestamp
 4. Write transformed data to Silver schema

In [0]:
dfRefunds = spark.table("asql_gizmobox_subbu_sql_catalog.dbo.refunds");
display(dfRefunds)

In [0]:
from pyspark.sql import functions as f
dfRefunds_UsingSplit = (
    dfRefunds.select(
        "refund_id",
        "payment_id",
        "refund_timestamp",
        "refund_amount",
        f.split("refund_reason", ":")[0].alias("refund_reason") ,
        f.split("refund_reason", ":")[1].alias("refund_source")
        )
)

display(dfRefunds_UsingSplit)

In [0]:
from pyspark.sql import functions as f

dfRefunds_final = (
    dfRefunds.select(
        "refund_id",
        "payment_id",
        f.date_format("refund_timestamp", "yyyy-MM-dd").cast("date").alias("refund_date"),
        f.date_format("refund_timestamp","HH:mm:ss").alias("refund_timestamp"),
        "refund_amount",
        f.regexp_extract('refund_reason','^([^:]+):',1).alias('refund_reason'),
        f.regexp_extract('refund_reason','^[^:]+:(.*)$',1).alias('refund_source')   
        )
)

display(dfRefunds_final)

In [0]:
dfRefunds_final.writeTo("gizmobox_sivan.silver.py_refunds").createOrReplace()

In [0]:
df = spark.table("gizmobox_sivan.silver.py_refunds")
display(df)